In [3]:
#Load anomaly dataset

In [5]:
import numpy as np
import pandas as pd


In [6]:
df = pd.read_csv("dataset_with_anomalies.csv")


In [8]:
# Identify numeric and categorical columns
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = df.select_dtypes(include=['object']).columns

# Handle numeric missing values
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Handle categorical missing values
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])


In [9]:
#Remove Duplicate Records

In [10]:
df.drop_duplicates(inplace=True)


In [11]:
#Fix Datatype Mismatches

In [13]:
# Convert columns to numeric safely
df['order_quantity'] = pd.to_numeric(df['order_quantity'], errors='coerce')
df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')

# Fill missing values by reassignment (✅ correct way)
df['order_quantity'] = df['order_quantity'].fillna(df['order_quantity'].median())
df['unit_price'] = df['unit_price'].fillna(df['unit_price'].median())


In [14]:
#Handle Outliers (IQR Method)

In [15]:
def remove_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return df[(df[column] >= lower) & (df[column] <= upper)]

df = remove_outliers_iqr(df, 'shipping_distance_km')
df = remove_outliers_iqr(df, 'unit_price')


In [16]:
#Fix Logical Errors (Dates)

In [17]:
# Convert to datetime
df['order_date'] = pd.to_datetime(df['order_date'])
df['actual_delivery_date'] = pd.to_datetime(df['actual_delivery_date'])

# Remove records where delivery happened before order
df = df[df['actual_delivery_date'] >= df['order_date']]


In [18]:
#Fix Incorrect Region Names

In [19]:
df['region'] = df['region'].replace({
    'Inda': 'India',
    'USAa': 'USA'
})


In [20]:
#Save Cleaned Dataset

In [21]:
df.to_csv("cleaned_dataset.csv", index=False)
